# Chronic Kidney Disease Dataset Preprocessing

Import Libraries

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

Load Dataset

In [2]:
datasetPath = "dataset/chronic-kidney-disease.csv"

# Load the dataset from the project dataset folder.
ckdData = pd.read_csv(datasetPath)

display(ckdData.head())
print(f"Dataset shape: {ckdData.shape}")

,patientId,age,gender,bmi,systolicBp,diastolicBp,serumCreatinine,egfr,bloodUrea,hemoglobin,urineProtein,urineSpecificGravity,bloodGlucose,diabetesMellitus,hypertension,smokingStatus,familyHistoryCkd,anemia,pedalEdema,ckdDiagnosis
0,1,62,Female,28.6,140,84,1.61,76.9,12.4,14.5,2,1.011,96.7,No,Yes,Former,No,No,No,No CKD
1,2,41,Female,34.6,105,78,1.41,58.5,30.6,12.3,2,1.011,121.7,No,No,Never,Yes,Yes,No,CKD
2,3,68,Female,26.4,115,80,2.35,66.4,7.0,12.8,2,1.019,75.8,No,No,Never,Yes,Yes,No,CKD
3,4,71,Male,25.5,132,79,2.55,104.9,44.2,12.4,3,1.006,105.8,No,No,Never,No,Yes,No,CKD
4,5,28,Female,29.9,130,55,0.82,96.1,7.0,14.0,1,1.023,65.0,No,No,Never,No,No,No,No CKD


Dataset shape: (46450, 20)


In [3]:
# Work on a copy to protect the original dataset.
processedData = ckdData.copy()

Remove Unnecessary Columns

In [4]:
# Remove patientId only
if "patientId" in processedData.columns:
    processedData = processedData.drop(columns=["patientId"])

processedData.head()

,age,gender,bmi,systolicBp,diastolicBp,serumCreatinine,egfr,bloodUrea,hemoglobin,urineProtein,urineSpecificGravity,bloodGlucose,diabetesMellitus,hypertension,smokingStatus,familyHistoryCkd,anemia,pedalEdema,ckdDiagnosis
0,62,Female,28.6,140,84,1.61,76.9,12.4,14.5,2,1.011,96.7,No,Yes,Former,No,No,No,No CKD
1,41,Female,34.6,105,78,1.41,58.5,30.6,12.3,2,1.011,121.7,No,No,Never,Yes,Yes,No,CKD
2,68,Female,26.4,115,80,2.35,66.4,7.0,12.8,2,1.019,75.8,No,No,Never,Yes,Yes,No,CKD
3,71,Male,25.5,132,79,2.55,104.9,44.2,12.4,3,1.006,105.8,No,No,Never,No,Yes,No,CKD
4,28,Female,29.9,130,55,0.82,96.1,7.0,14.0,1,1.023,65.0,No,No,Never,No,No,No,No CKD


Encode Target Variable (No CKD is represented as 0 and CKD is represented as 1) 

In [5]:
targetColumn = "ckdDiagnosis"
targetMapping = {"No CKD": 0, "CKD": 1}

# Encode the target variable using the required class mapping.
processedData[targetColumn] = processedData[targetColumn].map(targetMapping)

encodedTargetDistribution = (
    processedData[targetColumn]
    .value_counts()
    .sort_index()
    .rename_axis(targetColumn)
    .reset_index(name="frequency")
)

encodedTargetDistribution

,ckdDiagnosis,frequency
0,0,22136
1,1,24314


Encode Categorical Features

In [6]:
# Detect categorical predictor columns and exclude the target column.
categoricalColumns = processedData.select_dtypes(include=["object", "category"]).columns.tolist()
categoricalColumns = [columnName for columnName in categoricalColumns if columnName != targetColumn]

print("Categorical predictor columns:", categoricalColumns)

Categorical predictor columns: ['gender', 'diabetesMellitus', 'hypertension', 'smokingStatus', 'familyHistoryCkd', 'anemia', 'pedalEdema']


In [7]:
# Apply One-Hot Encoding to all categorical predictor variables.
processedData = pd.get_dummies(
    processedData,
    columns=categoricalColumns,
    drop_first=True
)

# Convert any boolean columns created by encoding into integers.
booleanColumns = processedData.select_dtypes(include=["bool"]).columns
processedData[booleanColumns] = processedData[booleanColumns].astype(int)


Split Features and Target Variable

In [8]:
# Split the processed data into features and target.
X = processedData.drop(columns=[targetColumn])
y = processedData[[targetColumn]]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (46450, 19)
y shape: (46450, 1)


Train-Test Split (80/20 stratified split)

In [9]:
# Use one shared train-test split for all models.
XTrain, XTest, yTrain, yTest = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"XTrain shape: {XTrain.shape}")
print(f"XTest shape: {XTest.shape}")
print(f"yTrain shape: {yTrain.shape}")
print(f"yTest shape: {yTest.shape}")

XTrain shape: (37160, 19)
XTest shape: (9290, 19)
yTrain shape: (37160, 1)
yTest shape: (9290, 1)


#### Feature Scaling for Logistic Regression

In [10]:
# Fit the scaler only on training data to avoid data leakage.
standardScaler = StandardScaler()

XTrainScaled = pd.DataFrame(
    standardScaler.fit_transform(XTrain),
    columns=XTrain.columns,
    index=XTrain.index
)

XTestScaled = pd.DataFrame(
    standardScaler.transform(XTest),
    columns=XTest.columns,
    index=XTest.index
)

print(f"XTrainScaled shape: {XTrainScaled.shape}")
print(f"XTestScaled shape: {XTestScaled.shape}")

XTrainScaled shape: (37160, 19)
XTestScaled shape: (9290, 19)


#### Save Processed Data

In [11]:
# Create the dataset folder if it does not already exist.
os.makedirs("dataset", exist_ok=True)

# Save unscaled features for Random Forest and XGBoost.
XTrain.to_csv("dataset/XTrain.csv", index=False)
XTest.to_csv("dataset/XTest.csv", index=False)

# Save scaled features for Logistic Regression.
XTrainScaled.to_csv("dataset/XTrainScaled.csv", index=False)
XTestScaled.to_csv("dataset/XTestScaled.csv", index=False)

# Save the shared target split.
yTrain.to_csv("dataset/yTrain.csv", index=False)
yTest.to_csv("dataset/yTest.csv", index=False)

print("Processed training and testing datasets saved successfully.")

Processed training and testing datasets saved successfully.
